# Análisis de outliers del dataset sintético

Detección y cuantificación de outliers por columna usando IQR y Z-score.

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("synthetic_api_activity.csv", parse_dates=["timestamp_local"])
df["hour"] = df["timestamp_local"].dt.hour
df["ratio_4xx"] = df["http_4xx"] / df["actividad"] * 100

numeric_cols = ["actividad", "http_4xx", "http_5xx", "sum_downloadTime", "avg_downloadTime", "ratio_4xx"]

## 1. Detección por IQR (método clásico)

Outlier = valor fuera de [Q1 - 1.5·IQR, Q3 + 1.5·IQR].

In [9]:
results = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    results.append({
        "columna": col,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "límite_inf": round(lower, 2),
        "límite_sup": round(upper, 2),
        "n_outliers": len(outliers),
        "pct_outliers": round(len(outliers) / len(df) * 100, 2)
    })

iqr_df = pd.DataFrame(results)
print(iqr_df.to_string(index=False))

         columna       Q1        Q3       IQR  límite_inf  límite_sup  n_outliers  pct_outliers
       actividad    92.00    696.00    604.00     -814.00     1602.00          37          0.14
        http_4xx     1.00      7.00      6.00       -8.00       16.00         514          1.98
        http_5xx     0.00      0.00      0.00        0.00        0.00          97          0.37
sum_downloadTime 21306.75 167806.25 146499.50  -198442.50   387555.50         105          0.41
avg_downloadTime   217.04    268.98     51.95      139.11      346.90         248          0.96
       ratio_4xx     0.55      1.55      1.00       -0.94        3.05        1063          4.10


## 2. Boxplots por columna

In [10]:
fig = make_subplots(rows=2, cols=3, subplot_titles=numeric_cols)
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for col, (r, c) in zip(numeric_cols, positions):
    fig.add_trace(go.Box(y=df[col], name=col, boxpoints="outliers",
                         marker=dict(opacity=0.3, size=2)), row=r, col=c)

fig.update_layout(height=600, showlegend=False, title_text="Boxplots - outliers por IQR")
fig.show()

## 3. Detección por Z-score

Outlier = |z| > 3 (más de 3 desviaciones estándar de la media).

In [11]:
results_z = []
for col in numeric_cols:
    z = (df[col] - df[col].mean()) / df[col].std()
    n_out = (z.abs() > 3).sum()
    results_z.append({
        "columna": col,
        "media": round(df[col].mean(), 2),
        "std": round(df[col].std(), 2),
        "n_outliers_z3": n_out,
        "pct_outliers_z3": round(n_out / len(df) * 100, 2)
    })

zscore_df = pd.DataFrame(results_z)
print(zscore_df.to_string(index=False))

         columna     media       std  n_outliers_z3  pct_outliers_z3
       actividad    406.79    346.93             42             0.16
        http_4xx      4.73      5.52             84             0.32
        http_5xx      0.47     10.77             70             0.27
sum_downloadTime 103969.62 109794.77             86             0.33
avg_downloadTime    246.56     75.76             89             0.34
       ratio_4xx      1.23      3.02             46             0.18


## 4. Outliers de actividad por franja horaria

IQR calculado por hora para respetar el patrón diurno. Así detectamos outliers *relativos a su franja*, no al global.

In [12]:
def count_outliers_by_group(group):
    q1 = group["actividad"].quantile(0.25)
    q3 = group["actividad"].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((group["actividad"] < lower) | (group["actividad"] > upper)).sum()
    return pd.Series({"n_outliers": n_out, "total": len(group), "pct": round(n_out / len(group) * 100, 2)})

by_hour = df.groupby("hour").apply(count_outliers_by_group).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=by_hour["hour"], y=by_hour["pct"], name="% outliers",
                     text=by_hour["n_outliers"], textposition="outside"))
fig.update_layout(title="Outliers de actividad por hora (IQR por franja)",
                  xaxis_title="Hora", yaxis_title="% outliers", height=400)
fig.show()

C:\Users\nicol\AppData\Local\Temp\ipykernel_26300\1640987624.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_hour = df.groupby("hour").apply(count_outliers_by_group).reset_index()


## 5. Localización temporal de outliers de actividad

Puntos normales en gris, outliers (IQR global) en rojo.

In [13]:
q1 = df["actividad"].quantile(0.25)
q3 = df["actividad"].quantile(0.75)
iqr = q3 - q1
df["is_outlier_act"] = (df["actividad"] < q1 - 1.5 * iqr) | (df["actividad"] > q3 + 1.5 * iqr)

fig = go.Figure()
normal = df[~df["is_outlier_act"]]
outliers = df[df["is_outlier_act"]]

fig.add_trace(go.Scattergl(x=normal["timestamp_local"], y=normal["actividad"],
                            mode="markers", marker=dict(size=1.5, color="lightgray"),
                            name=f"Normal ({len(normal)})"))
fig.add_trace(go.Scattergl(x=outliers["timestamp_local"], y=outliers["actividad"],
                            mode="markers", marker=dict(size=4, color="red"),
                            name=f"Outliers ({len(outliers)})"))

fig.update_layout(title="Outliers de actividad (IQR global)", height=450,
                  xaxis_title="Fecha", yaxis_title="Actividad")
fig.show()

print(f"Total outliers actividad: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

Total outliers actividad: 37 (0.14%)


## 6. Comparativa IQR vs Z-score

In [14]:
comp = iqr_df[["columna", "n_outliers", "pct_outliers"]].merge(
    zscore_df[["columna", "n_outliers_z3", "pct_outliers_z3"]], on="columna")
comp.columns = ["columna", "n_IQR", "pct_IQR", "n_Zscore", "pct_Zscore"]
print(comp.to_string(index=False))

         columna  n_IQR  pct_IQR  n_Zscore  pct_Zscore
       actividad     37     0.14        42        0.16
        http_4xx    514     1.98        84        0.32
        http_5xx     97     0.37        70        0.27
sum_downloadTime    105     0.41        86        0.33
avg_downloadTime    248     0.96        89        0.34
       ratio_4xx   1063     4.10        46        0.18
